# Lesson 1 - Creating a single WRF-Hydro model run through PEST++

The easiest way to check whether the coupling between the model and the PEST++ works as it should is to create a single model run. This single model run will later be used in Lesson 6 for weighting of the observations and building the objective function. To run PEST++ with WRF-Hydro, one would need to create a folder that has all the files from both systems plus pre and post-processing scripts for communicating the information between the two systems. You already got familiar with the required files for running WRF-Hydro model in Lesson 2, and we assume you have familiarity with the PEST++, we will quickly mention the files that are used in this training below. The pre and post processing scripts for running WRF-Hydro also has been provided here, and will be explained further below. 

In [1]:
# load the required libraries 
import os 
import pyemu 
import pandas as pd

## **Step 1. Create a work directory and copy all the required information** 

This folder needs to have all the required information for running PEST++/WRF-Hydro which included the followings: 
* PEST++ binary files 
* PEST++ necessary files (control file, template file, instruction file) 
* WRF-Hydro executable and TBL files  
* WRF-Hydro example case (static files, forcing and namelists)
* WRF-Hydro pre and postprocessing scripts required to communicate with PEST++

In [2]:
%%bash
# create the working directory
mkdir -p ~/wrf_hydro_training/output/pestpp/lesson1/Single_Model_Run

## **Step 2. Copy WRF-Hydro related files**

We will create a folder named `WRFHydro_Model` and copy the required WRF-Hydro model in there. 
We start with copying the WRF-Hydro executable and the TABLE files from the *~/wrf-hydro-training/wrf_hydro_nwm_public/trunk/NDHMS/Run* directory. These files are small so we will make actual copies rather than symbolic links in this case. 

In [3]:
%%bash

cd ~/wrf_hydro_training/output/pestpp/lesson1/Single_Model_Run

# create a new folder to hold all the WRF-Hydro files
mkdir WRFHydro_Model
cd WRFHydro_Model

# copy the wrf_hydro executable and the 
cp ~/wrf_hydro_training/wrf_hydro_nwm_public/build/Run/*.TBL .
cp ~/wrf_hydro_training/wrf_hydro_nwm_public/build/Run/wrf_hydro.exe .

ls 

CHANPARM.TBL
GENPARM.TBL
HYDRO.TBL
MPTABLE.TBL
SOILPARM.TBL
wrf_hydro.exe


Next we need to copy all the specific domain information such as static input files (`DOMAIN`), the atmospheric forcing files (`FORCING`), the model restart files to have a warm start simulations (`RESTART`) and the namelits provided with the example case. We place all the required files for the WRFHydro execution in a folder called WRFHydro_Model. 

In [4]:
%%bash 
# Let's copy the example case files
cd ~/wrf_hydro_training/output/pestpp/lesson1/Single_Model_Run/WRFHydro_Model
ln -sf ~/wrf_hydro_training/example_case/calibration/PESTPP/FORCING .
ln -sf ~/wrf_hydro_training/example_case/calibration/PESTPP/DOMAIN .
ln -sf ~/wrf_hydro_training/example_case/calibration/PESTPP/RESTARTS .

# copy namelist 
cp ~/wrf_hydro_training/example_case/calibration/PESTPP/namelist.hrldas .
cp ~/wrf_hydro_training/example_case/calibration/PESTPP/hydro.namelist .
ls 

CHANPARM.TBL
DOMAIN
FORCING
GENPARM.TBL
hydro.namelist
HYDRO.TBL
MPTABLE.TBL
namelist.hrldas
RESTARTS
SOILPARM.TBL
wrf_hydro.exe


## **Step 3. Copy PEST++ binary file**

To run PEST++, one would need to either compile PEST++ or use the available binary files provided with the code release. The binary files work on many systems. Here we have included the PEST++ binary files in the training material, and will copy it to the working directory. From the available binaries, we will use the `pestpp-ies` here, since we would like to perform the history matching using the iterative Ensemble Smoother (iES) technique. 


In [5]:
%%bash 
# Move one directory up, to the working directory
cd ~/wrf_hydro_training/output/pestpp/lesson1/Single_Model_Run

# copy the PEST++ binary files 
cp ~/wrf_hydro_training/pestpp/PESTPP_Bin/pestpp-ies .

## **Step 4. Copy all WRF-Hydro/PESTPP files**

We have prepared the files that matches the example case provided here, and this includes both the files required by PEST++ (control file, instruction and template files) and the scripts for running the WRF-Hydro Pre-processing, execution and post processing.

In [6]:
%%bash 
cd ~/wrf_hydro_training/output/pestpp/lesson1/Single_Model_Run

# copy all the pestpp required files 
cp ~/wrf_hydro_training/example_case/calibration/PESTPP/PESTPP/* .
ls

calib_parameters.csv
inputfile.csv
outputfile.csv
pestpp-ies
post_processing_WRFHydro_run.py
pre_processing_WRFHydro_run.py
runCmd.sh
WRFHydro_Model
wrfpif.ins
wrfpst.pargp_data.csv
wrfpst.pst
wrftpl.tpl


let take a look at the content of the files provided. Programs of both the PEST and PEST++ suites require three types of input file. These are:
* template files, one for each model input file in which parameters or decision variables reside. Here `wrftpl.tpl` file is the template file prepared for WRF-Hydro. The PEST++ workflow will write the parameters in this file, and the `pre_processing_WRFHydro_run.py` file will read that file and apply the new set of parameters to the netcdf files. 
* instruction files, one for each model output file from which numbers must be read. Here `wrfpif.ins` file is the instruction file prepared for the WRF-Hydro. The `post_processing_WRFHydro_run.py` script should create a file with similar format and fill in the simulated streamflow values in place of entries such as `!obs_20180801_010000!`. 
* a control file, supplying the PEST or PEST++ program with the names of all template and instruction files, the names of corresponding model input and output files, the values of control variables, initial parameter values, measurement values, weights, etc. Here file `wrfpst.pst` file is the control file used for the training. 

In [7]:
%%bash
cat ~/wrf_hydro_training/output/pestpp/lesson1/Single_Model_Run/wrftpl.tpl

ptf ~
bexp           ~bexp~
smcmax         ~smcmax~
dksat          ~dksat~
refkdt         ~refkdt~
slope          ~slope~
retdeprtfac    ~retdeprtfac~
lksatfac       ~lksatfac~
zmax           ~zmax~
expon          ~expon~
Coeff          ~Coeff~
cwpvt          ~cwpvt~
vcmx25         ~vcmx25~
mp             ~mp~
hvt            ~hvt~ 
mfsno          ~mfsno~
rsurfexp       ~rsurfexp~	 
nexp           ~nexp~
AXAJ           ~AXAJ~
BXAJ           ~BXAJ~
XXAJ           ~XXAJ~
Loss           ~Loss~
rsurfsnow      ~rsurfsnow~
scamax         ~scamax~


In [8]:
%%bash
cat ~/wrf_hydro_training/output/pestpp/lesson1/Single_Model_Run/wrfpif.ins | head -n 10

pif @ 
 @obsnme@ 
l1 w !kge! w w 
l1 w !obs_20180801_010000! w w 
l1 w !obs_20180801_020000! w w 
l1 w !obs_20180801_030000! w w 
l1 w !obs_20180801_040000! w w 
l1 w !obs_20180801_050000! w w 
l1 w !obs_20180801_060000! w w 
l1 w !obs_20180801_070000! w w 


In [9]:
%%bash
cat ~/wrf_hydro_training/output/pestpp/lesson1/Single_Model_Run/wrfpst.pst

pcf version=2
* control data keyword
pestmode                                 estimation
noptmax                                 0
svdmode                                 1
maxsing                                50
eigthresh                           1e-06
eigwrite                                1
ies_num_reals                  1
ies_bad_phi_sigma              2.5
overdue_giveup_fac             4
ies_save_rescov                false
ies_no_noise                   true
ies_drop_conflicts             false
ies_pdc_sigma_distance         3.0
ies_num_threads                8
ies_lambda_mults               1.0
lambda_scale_fac               1.0
ies_subset_size                10
panther_echo                   false
* parameter groups external
wrfpst.pargp_data.csv
* parameter data external
calib_parameters.csv
* observation data external
/glade/u/home/arezoo/wrf_hydro_training/example_case/calibration/PESTPP/OBS/Observation_File_20180801_20180901.csv
* model command line
sh runCmd.sh
* model

### **calib_parameters.csv**: 
This file contains a subset of parameters that we could calibrate in WRF-Hydro. If one wish to calibrate any other parameters than what is provided in this table, 
we should make appropriate modifications to the `pre_processing_WRFHydro_run.py` script to make sure the parameter is changed in the appropriate WRF-Hydro file before running WRF-Hydro model. 

In [10]:
calib_parameters = pd.read_csv("~/wrf_hydro_training/output/pestpp/lesson1/Single_Model_Run/calib_parameters.csv")
calib_parameters

,parnme,partrans,parchglim,parval1,parlbnd,parubnd,pargp,scale,offset,dercom
0,bexp,none,relative,1.000,0.4000,1.9,group1,1,0,1
1,smcmax,none,relative,1.000,0.8000,1.2,group1,1,0,1
2,dksat,none,relative,1.000,0.2000,10.0,group1,1,0,1
3,refkdt,fixed,relative,1.000,0.1000,4.0,group1,1,0,1
4,slope,none,relative,0.330,0.0000,1.0,group1,1,0,1
5,retdeprtfac,none,relative,1.000,0.1000,20000.0,group1,1,0,1
6,lksatfac,none,relative,1000.000,10.0000,10000.0,group1,1,0,1
7,zmax,none,relative,50.000,10.0000,250.0,group1,1,0,1
8,expon,none,relative,3.000,1.0000,8.0,group1,1,0,1
9,Coeff,fixed,relative,0.001,0.0001,0.1,group1,1,0,1


`inputfile.csv` and `outputfile.csv` has the name of the files required by PEST++ and the WRF-Hydro for both parameters and the model observation/simulations. 

In [11]:
%%bash
cat ~/wrf_hydro_training/output/pestpp/lesson1/Single_Model_Run//inputfile.csv
cat ~/wrf_hydro_training/output/pestpp/lesson1/Single_Model_Run//outputfile.csv

pest_file,model_file
wrftpl.tpl,wrftpl_input.txt
pest_file,model_file
wrfpif.ins,wrfpif.txt


### How to run the model within PEST++ 
In the control file, there is an option `model command line` which user could specify the command that will run the model. Here we are using a shell script called `runCmd.sh` which does three things: 
* run the pre-processing script which takes the parameters from the `wrftpl_input.txt` file, and apply the parameters to the netcdf files. 
* run the WRF-Hydro model. 
* run the post-processing script which will write the streamflow simulation in a files called `wrfpif.txt` a mirror of the `wrfpif.ins` with the values. 


In [12]:
%%bash
cat ~/wrf_hydro_training/output/pestpp/lesson1/Single_Model_Run/runCmd.sh

#!/bin/bash
# script for running the pre and post processing a WRF-Hydro model run 

# pre-processing 
python pre_processing_WRFHydro_run.py ./WRFHydro_Model ~/wrf_hydro_training/example_case/calibration/PESTPP/DOMAIN wrftpl_input.txt 4 0 0 &  
wait

cd WRFHydro_Model

# running the model 
./wrf_hydro.exe  &> wrfoutput.out &
wait

cd .. 

# post-processing 
python post_processing_WRFHydro_run.py 01473000 4782107 ./WRFHydro_Model ~/wrf_hydro_training/example_case/calibration/PESTPP/OBS ./wrfpif.txt "2018-08-10 00:00" & 
wait 


## **Step 5. Make modifications to the files**

Let s modify the namelists to read the netcdf files from the WRF-Hydro run directory instead of the DOMAIN directory, since we make modification to these files. The pre-processing script will read the parameters from the PEST++ output files and then modify the input netcdf files to WRF-Hydro. 

In [13]:
%%bash 
cd ~/wrf_hydro_training/output/pestpp/lesson1/Single_Model_Run/WRFHydro_Model/
# Edit namelist.hrldas 
sed -i -e 's#DOMAIN/soil_properties.nc#soil_properties.nc#g' namelist.hrldas
sed -i -e 's#DOMAIN/Fulldom.nc#Fulldom.nc#g' hydro.namelist
sed -i -e 's#DOMAIN/HYDRO_TBL_2D.nc#HYDRO_TBL_2D.nc#g' hydro.namelist
sed -i -e 's#DOMAIN/GWBUCKPARM.nc#GWBUCKPARM.nc#g' hydro.namelist

## **Step 6. Running PEST++/WRF-Hydro**

For the sake of this training, we will use a pyemy function called `utils.os_utils.start_workers` to run PEST++, which is suitable for training given the size of the experiment. Please refer to the PEST++ documentation for more information on this function (https://pyemu.readthedocs.io/en/develop/autoapi/pyemu/utils/os_utils/index.html). 


In [14]:
#  run PEST++
pyemu.utils.os_utils.start_workers(worker_dir = "/glade/u/home/arezoo/wrf_hydro_training/output/pestpp/lesson1/Single_Model_Run", 
                                   exe_rel_path = "pestpp-ies", 
                                   pst_rel_path = "wrfpst.pst", 
                                   num_workers=1, 
                                   worker_root='/glade/u/home/arezoo/wrf_hydro_training/output/pestpp/lesson1',
                                   master_dir = "/glade/u/home/arezoo/wrf_hydro_training/output/pestpp/lesson1/host",
                                   port=4004, 
                                   verbose = True, 
                                   cleanup = False)


master:./pestpp-ies wrfpst.pst /h :4004 in /glade/u/home/arezoo/wrf_hydro_training/output/pestpp/lesson1/host


             pestpp-ies: a GLM iterative ensemble smoother

                   by the PEST++ development team


version: 5.2.9
binary compiled on Jan 18 2024 at 16:01:42

started at 10/29/26 15:35:58
...processing command line: ' ./pestpp-ies wrfpst.pst /h :4004'
...using panther run manager in master mode using port 4004

using control file: "wrfpst.pst"
in directory: "/glade/work/arezoo/USGS_NY_Flood/wrf_hydro_training/output/pestpp/lesson1/host"
on host: "crhtc18"

processing control file wrfpst.pst

Note: 'NOPTMAX' == 0, switching to forgiveness mode when checking inputs

noptmax = 0, resetting max_run_fail = 1





:~-._                                                 _.-~:
: :.~^o._        ________---------________        _.o^~.:.:
 : ::.`?88booo~~~.::::::::...::::::::::::..~~oood88P'.::.:
 :  ::: `?88P .:::....         ........:::::. ?88P' :::. :
  :  :::. `? .::.     

## **Step 7. Let's check the outputs**

Let s check the host directory. 

In [15]:
%%bash 
ls  /glade/u/home/arezoo/wrf_hydro_training/output/pestpp/lesson1/host

calib_parameters.csv
inputfile.csv
outputfile.csv
pestpp-ies
post_processing_WRFHydro_run.py
pre_processing_WRFHydro_run.py
runCmd.sh
WRFHydro_Model
wrfpif.ins
wrfpst.base.obs.csv
wrfpst.base.par
wrfpst.base.rei
wrfpst.log
wrfpst.pargp_data.csv
wrfpst.phi.actual.csv
wrfpst.phi.composite.csv
wrfpst.phi.group.csv
wrfpst.phi.meas.csv
wrfpst.phi.regul.csv
wrfpst.pst
wrfpst.rec
wrfpst.rmr
wrftpl.tpl


The successful finish message of PEST++ could be found in the `wrfpst.rmr` file. 

In [16]:
%%bash 
cat  /glade/u/home/arezoo/wrf_hydro_training/output/pestpp/lesson1/host/wrfpst.rec | tail -n 5


pestpp-ies analysis complete...
started at 10/29/26 15:35:58
finished at 10/29/26 15:37:50
took 1.86667 minutes


Let us check the content of the workder folder. 

In [17]:
%%bash 
ls  /glade/u/home/arezoo/wrf_hydro_training/output/pestpp/lesson1/worker_0/

calib_parameters.csv
inputfile.csv
Observation_File.csv
outputfile.csv
panther_worker.rec
pestpp-ies
post_processing_WRFHydro_run.py
pre_processing_WRFHydro_run.py
runCmd.sh
run.info
WRFHydro_Model
wrfpif.ins
wrfpif.ins_workflow_generated
wrfpif.txt
wrfpif.txtbackup
wrfpst.pargp_data.csv
wrfpst.pst
wrftpl_input.txt
wrftpl.tpl


Now we could continue to the next lesson where we submit a Monte Carlo model simulation using PEST++. 